# 52. PDF 표면 열접촉 조건부 신호 — 3개 무작위 설정 앙상블
51번 온도 관계 신호를 RF와 ExtraTrees 각각 seed42/7/101 평균으로 확인한다. 두 트리와 로지스틱3설정 평균을 결합한다. 정상 참조30개와 분위수 후보를 그대로 고정한다. 새 검증 데이터가 아니고 가스 세부유형·원인 확정이 아니다.
원피처6개: 배럴–금형 온도차, 금형3−4, 금형 변화 평균, 과거10샷 금형 편차 평균, 배럴5변화−금형변화, 최고사출압/충진시간. 정상 참조 조건에 배럴5·최대속도·충진시간·호퍼를 쓰므로 총 입력 센서가6개뿐이라는 뜻은 아니다.


In [ ]:
from pathlib import Path
import json,time
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
anchor=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv',parse_dates=['TimeStamp'])
anchor=anchor[anchor.model.eq('three_seed_mean')].set_index(['scheme','TimeStamp'])
oldinner=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
# Mechanism names are hypotheses, not gold-standard causal labels.
p=d.Max_Injection_Pressure
sw=d.Max_Switch_Over_Pressure
mold=(d.Mold_Temperature_3+d.Mold_Temperature_4)/2
barrel=d[[f'Barrel_Temperature_{i}' for i in range(1,7)]].mean(axis=1)
def ratio(a,b):return a/b.where(b.abs()>1e-6)
F={
 'pressure_support':pd.DataFrame({
 'peak_minus_switch':p-sw,
 'switch_fraction':ratio(sw,p),
 'cushion_per_meter_position':ratio(d.Cushion_Position,d.Plasticizing_Position),
 'pressure_dev10':d.Max_Injection_Pressure__dev10,
 'switch_pressure_delta':d.Max_Switch_Over_Pressure__delta1,
 'cushion_delta':d.Cushion_Position__delta1}),
 'surface_thermal_contact':pd.DataFrame({
 'barrel_mold_gap':barrel-mold,
 'mold_side_difference':d.Mold_Temperature_3-d.Mold_Temperature_4,
 'mold_delta_mean':(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'mold_dev_mean':(d.Mold_Temperature_3__dev10+d.Mold_Temperature_4__dev10)/2,
 'barrel_mold_delta_gap':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'pressure_per_filling_time':ratio(p,d.Filling_Time)}),
 'material_release_proxy':pd.DataFrame({
 'barrel_spatial_spread':d[[f'Barrel_Temperature_{i}' for i in range(1,7)]].std(axis=1),
 'barrel6_minus5':d.Barrel_Temperature_6-d.Barrel_Temperature_5,
 'hopper_barrel_gap':barrel-d.Hopper_Temperature,
 'cycle_minus_plasticizing':d.Cycle_Time-d.Plasticizing_Time,
 'plasticizing_fraction':ratio(d.Plasticizing_Time,d.Cycle_Time),
 'back_pressure_dev10':d.Average_Back_Pressure__dev10})
}
F={'surface_thermal_contact':F['surface_thermal_contact']}
for name,x in F.items():
 x.replace([np.inf,-np.inf],np.nan,inplace=True)
 x.assign(TimeStamp=d.TimeStamp).to_csv(tables/f'rg3_pdf_{name}_features.csv',index=False,encoding='utf-8-sig')
 print('Feature group',name,list(x.columns),flush=True)
from sklearn.neighbors import NearestNeighbors
controls={
 'pressure_support':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3'],
 'surface_thermal_contact':['Barrel_Temperature_5','Max_Injection_Speed','Filling_Time','Hopper_Temperature'],
 'material_release_proxy':['Plasticizing_Position','Max_Injection_Speed','Mold_Temperature_3','Average_Back_Pressure']
}
def local_residual(ref,q,name):
 normal=ref[y[ref]==0]
 raw=F[name].to_numpy(float)
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(raw[normal])
 raw=imp.transform(raw)
 context=d[controls[name]].to_numpy(float)
 ci=SimpleImputer(strategy='median',keep_empty_features=True).fit(context[normal])
 context=ci.transform(context)
 sc=RobustScaler().fit(context[normal]);context=sc.transform(context)
 k=min(31,len(normal))
 near=NearestNeighbors(n_neighbors=k).fit(context[normal])
 distances,neighbors=near.kneighbors(context[q])
 global_scale=np.std(raw[normal],axis=0)
 global_scale=np.maximum(global_scale*.1,1e-6)
 out=[]
 for pos,i in enumerate(q):
  indexes=normal[neighbors[pos]]
  indexes=indexes[indexes!=i][:30]
  if len(indexes)==0:indexes=normal
  center=np.median(raw[indexes],axis=0)
  scale=np.maximum(np.std(raw[indexes],axis=0),global_scale)
  out.append((raw[i]-center)/scale)
 return np.asarray(out)
memo={}
def predict(ref,q,name):
 key=(tuple(ref),tuple(q),name)
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),4))
 imp=SimpleImputer(strategy='median',keep_empty_features=True)
 tr=imp.fit_transform(local_residual(ref,ref,name));te=imp.transform(local_residual(ref,q,name))
 tree=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  seeds=[]
  for seed in [42,7,101]:
   model=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
   seeds.append(model.fit(tr,y[ref]).predict_proba(te)[:,1])
  tree.append(np.mean(seeds,axis=0))
 # Fit scaling only on normals from the reference training data.
 sc=RobustScaler().fit(tr[y[ref]==0]);tx=sc.transform(tr);qx=sc.transform(te)
 linear=[]
 for C in [.1,1.,10.]:
  model=LogisticRegression(C=C,class_weight='balanced',max_iter=2000,random_state=42)
  linear.append(model.fit(tx,y[ref]).predict_proba(qx)[:,1])
 a,b=tree;c=np.mean(linear,axis=0)
 out=np.column_stack([(a+b)/2,(a+b+c)/3,np.sqrt(np.clip((a+b)/2*c,0,1)),c])
 memo[key]=out
 return out
models=['two_tree_mean','tree_linear_mean','tree_linear_geometric','logit3']
records=[];choices=[];audits=[];innerrows=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
 for j,block in enumerate(blocks):
  q=np.flatnonzero(groups.eq(block))
  ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
  if len(np.unique(y[ref]))<2:continue
  if scheme=='past_blocks_only':assert d.TimeStamp.iloc[ref].max()<d.TimeStamp.iloc[q].min()
  anc=anchor.loc[(scheme,d.TimeStamp.iloc[q]),'prediction'].to_numpy(bool)
  base=anchor.loc[(scheme,d.TimeStamp.iloc[q]),'baseline_prediction'].to_numpy(bool)
  ic=oldinner.loc[oldinner.scheme.eq(scheme)&oldinner.heldout_block.eq(block)].set_index('TimeStamp').loc[d.TimeStamp.iloc[ref]]
  oldcut=float(np.quantile(ic.loc[y[ref]==0,'raw_baseline_score'],.99,method='higher'))
  oldpr=ic.raw_baseline_score.gt(oldcut).to_numpy()
  tn0,fp0,fn0,tp0=confusion_matrix(y[ref],oldpr,labels=[0,1]).ravel()
  f0=2*tp0/(2*tp0+fp0+fn0) if 2*tp0+fp0+fn0 else 0
  candidates=[]
  for name in F:
   outer=predict(ref,q,name)
   inner=np.full((len(ref),4),np.nan)
   for ib in groups.iloc[ref].unique():
    mask=groups.iloc[ref].eq(ib).to_numpy()
    inner[mask]=predict(ref[~mask],ref[mask],name)
   for k,model in enumerate(models):
    norm=inner[(y[ref]==0)&np.isfinite(inner[:,k]),k]
    for rate in [.005,.01,.025]:
     cut=float(np.quantile(norm,1-rate,method='higher')) if len(norm) else 1.
     valid=np.isfinite(inner[:,k]);pr=(oldpr|(inner[:,k]>cut))[valid]
     tn,fp,fn,tp=confusion_matrix(y[ref][valid],pr,labels=[0,1]).ravel()
     f=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
     allow=bool(valid.all() and tp>tp0 and fp-fp0<=2 and f>f0)
     row=dict(scheme=scheme,block=block,feature_group=name,model=model,budget=rate,threshold=cut,inner_TP=int(tp),inner_FP=int(fp),inner_F1=f,added_TP=int(tp-tp0),added_FP=int(fp-fp0),allowed=allow,baseline_inner_F1=f0)
     innerrows.append(row)
     candidates.append((row,outer[:,k]))
     extra=outer[:,k]>cut
     for combine,out in [('new_only',extra),('OR_anchor',anc|extra)]:
      for l,i in enumerate(q):
       records.append(dict(scheme=scheme,model=name+'__'+model,budget=rate,combine=combine,TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),prediction=int(out[l]),anchor_prediction=int(anc[l]),baseline_prediction=int(base[l]),score=float(outer[l,k]),threshold=cut))
  allowed=[c for c in candidates if c[0]['allowed']]
  # Select only on inner reference labels; never rank candidates by outer F1.
  best=max(allowed,key=lambda c:(c[0]['inner_F1'],-c[0]['added_FP'],-c[0]['budget'])) if allowed else None
  chosen=anc if best is None else anc|(best[1]>best[0]['threshold'])
  choices.append(dict(scheme=scheme,block=block,selected='none' if best is None else best[0]['feature_group']+'__'+best[0]['model'],budget=np.nan if best is None else best[0]['budget'],inner_added_TP=0 if best is None else best[0]['added_TP'],inner_added_FP=0 if best is None else best[0]['added_FP']))
  for l,i in enumerate(q):
   for name,out in [('anchor47',anc),('baseline_RF',base),('pdf_thermal_seed_guarded_selector',chosen)]:
    records.append(dict(scheme=scheme,model=name,budget=0.,combine='fixed',TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),prediction=int(out[l]),anchor_prediction=int(anc[l]),baseline_prediction=int(base[l]),score=np.nan,threshold=np.nan))
  audits.append(dict(scheme=scheme,block=block,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum()),time_overlap=len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q]))))
  print('Finished thermal seed ensemble',scheme,block,flush=True)
pp=pd.DataFrame(records)
metrics=[]
for (scheme,model,rate,combine),g in pp.groupby(['scheme','model','budget','combine'],sort=False):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 metrics.append(dict(scheme=scheme,model=model,budget=rate,combine=combine,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn) if tp+fn else 0.))
mm=pd.DataFrame(metrics)
for obj,name in [(pp,'predictions'),(mm,'metrics'),(pd.DataFrame(choices),'choices'),(pd.DataFrame(innerrows),'inner_candidates'),(pd.DataFrame(audits),'fold_audit')]:
 obj.to_csv(tables/f'rg3_pdf_thermal_seed_{name}.csv',index=False,encoding='utf-8-sig')
print('PREDECLARED GUARDED SELECTOR',mm[mm['combine'].eq('fixed')].to_string(index=False),flush=True)
print('EXPLORATORY ONLY: best outer results are not independent confirmation')
print(mm[mm['combine'].eq('OR_anchor')].sort_values(['scheme','F1','FP'],ascending=[True,False,True]).groupby('scheme').head(8).to_string(index=False))


